# Phase 5 v2 — Model Comparison on 234-dialog frozen benchmark

**Dataset**: 234 dialogs / ~144 min audio / 13 slices / 4 splits
  - dev_smoke 24 (codex) / dev_tune 30 (codex) / test_core 140 (codex 120 + F+F extension 20) / role_attribution 40 (ours)
  - **20 cases have 3 speakers** (`third_party_background` 17 + `impostor_unknown` 3); rest have 2

**Models** (3 configs — community-1/ERes2NetV2/Sortformer dropped):
1. pyannote 3.1 (auto speaker count)
2. pyannote 3.1 (**fixed N per spec** — uses 2 or 3 depending on case, not hardcoded 2)
3. 3D-Speaker CAM++

**Dropped & reasons**:
- pyannote community-1: requires pyannote.audio >= 3.5 (we pin 3.4.0), config has `plda` param not in 3.4
- ERes2NetV2: monkey-patch worked in spot test but user decided not worth complexity for KGI prod
- NVIDIA Sortformer v1: weak on Mandarin/Hokkien (v1 results showed 27%+ DER)

**Pre-requisites** (already done locally):
1. `scripts/generate_audio_and_gt.py` produced audio + GT for all 234
2. Upload to Drive at `KGI_STT/phase5_v2/`:
   - `audio/dialogs_clean/` (234 wav, 264MB)
   - `data/ground_truth_per_case/{speech,turn}/` (234 rttm each)
   - `data/dialog_specs/all_dialogs.json` (single canonical, 1MB)

**Strict integrity** (setup cell asserts):
- exactly 234 wav / 234 speech.rttm / 234 turn.rttm
- case_id set == `all_dialogs.json` (no missing, no extra)
- detection cache cleared at start (prevents v2_smk010/011-style contamination)

**Expected runtime**: ~50-70 min (3 models × 234 cases)


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, sys, json, shutil, datetime
BASE = pathlib.Path('/content/drive/MyDrive/KGI_STT/phase5_v2')
assert BASE.exists(), f'Drive folder not found: {BASE}. 先把本地 phase5_v2/ 上傳到 Drive。'

AUDIO_DIR    = BASE / 'audio' / 'dialogs_clean'
GT_SPEECH    = BASE / 'data' / 'ground_truth_per_case' / 'speech'
GT_TURN      = BASE / 'data' / 'ground_truth_per_case' / 'turn'
SPECS_DIR    = BASE / 'data' / 'dialog_specs'
RESULTS_DIR  = BASE / 'results'
MODELS_CACHE = BASE / 'models_cache'

# === RUN_ID ensures every benchmark run is isolated ===
RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
DETECT_DIR = RESULTS_DIR / 'detections'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_CACHE.mkdir(parents=True, exist_ok=True)

# === AUTO-ARCHIVE any pre-existing detections / metrics ===
# No prompt — staleness is fatal in this notebook. Old data preserved under _old_runs/.
old_runs = RESULTS_DIR / '_old_runs'
archive_needed = (DETECT_DIR.exists() and any(DETECT_DIR.iterdir())) or list(RESULTS_DIR.glob('metrics_*.csv'))
if archive_needed:
    archive_dir = old_runs / f'archived_before_{RUN_ID}'
    archive_dir.mkdir(parents=True, exist_ok=True)
    if DETECT_DIR.exists() and any(DETECT_DIR.iterdir()):
        shutil.move(str(DETECT_DIR), str(archive_dir / 'detections'))
        print(f'⚠️  Archived old detections → {archive_dir / "detections"}')
    for csv_f in list(RESULTS_DIR.glob('metrics_*.csv')) + list(RESULTS_DIR.glob('*_summary.csv')) + list(RESULTS_DIR.glob('der_by_*.csv')):
        shutil.move(str(csv_f), archive_dir / csv_f.name)
        print(f'⚠️  Archived old {csv_f.name}')
DETECT_DIR.mkdir(parents=True, exist_ok=True)
assert not any(DETECT_DIR.iterdir()), 'detections dir must be empty before run starts'
print(f'\nRUN_ID = {RUN_ID}, fresh detections dir confirmed empty')

# === Load canonical spec FIRST ===
spec_path = SPECS_DIR / 'all_dialogs.json'
assert spec_path.exists(), f'{spec_path} missing — should be the single source of truth.'
all_specs = json.loads(spec_path.read_text(encoding='utf-8'))
EXPECTED_CASE_IDS = {s['case_id'] for s in all_specs}
DIALOG_META = {s['case_id']: {
    'slice': s['slice'],
    'language': s['language_profile'],
    'expected_n_speakers': len(s['participants']),
} for s in all_specs}
EXPECTED_N = len(all_specs)
print(f'Spec: {EXPECTED_N} cases loaded from {spec_path.name}')

# === STRICT assertion: exact-count + exact case_id set ===
n_audio = len(list(AUDIO_DIR.glob('*.wav')))
n_speech_gt = len(list(GT_SPEECH.glob('*.rttm')))
n_turn_gt = len(list(GT_TURN.glob('*.rttm')))
print(f'On Drive: {n_audio} wav / {n_speech_gt} speech.rttm / {n_turn_gt} turn.rttm')

assert n_audio == EXPECTED_N, f'Expected exactly {EXPECTED_N} wav, found {n_audio}. Upload incomplete?'
assert n_speech_gt == EXPECTED_N, f'Expected {EXPECTED_N} speech rttm, found {n_speech_gt}'
assert n_turn_gt == EXPECTED_N, f'Expected {EXPECTED_N} turn rttm, found {n_turn_gt}'

audio_ids  = {p.stem for p in AUDIO_DIR.glob('*.wav')}
speech_ids = {p.stem for p in GT_SPEECH.glob('*.rttm')}
turn_ids   = {p.stem for p in GT_TURN.glob('*.rttm')}
for label, ids in [('audio', audio_ids), ('speech_gt', speech_ids), ('turn_gt', turn_ids)]:
    missing = EXPECTED_CASE_IDS - ids
    extra   = ids - EXPECTED_CASE_IDS
    assert not missing, f'{label} missing case_ids: {sorted(missing)[:10]}'
    assert not extra,   f'{label} has extra case_ids not in spec: {sorted(extra)[:10]}'

all_case_ids = sorted(EXPECTED_CASE_IDS)
print(f'✓ Strict integrity OK: {len(all_case_ids)} cases match spec exactly')

In [ ]:
# === API key (only HF for pyannote, no TTS needed) ===
HF_TOKEN = ''   # ← 你的 HuggingFace read token
assert HF_TOKEN, 'Set HF_TOKEN above'
print('HF token set ✓')

In [ ]:
# === STEP 1: Clean install — run ONCE, then Runtime → Restart session ===
# 3 models only: pyannote 3.1 (auto+fixed-N) + 3D-Speaker CAM++ via modelscope
# NO NeMo (Sortformer dropped) / NO 3D-Speaker repo clone (ERes2NetV2 dropped)
# Versions pinned for reproducibility — known-working combo from prior session

!pip install -q --upgrade pip

# 1) Uninstall conflicting packages so binary versions don't mismatch
!pip uninstall -y -q numpy scipy pandas torch torchvision torchaudio torchcodec pyannote.audio pyannote.metrics modelscope funasr 2>&1 | tail -3

# 2) Core torch/pyannote stack — pinned + --no-cache-dir for fresh wheels
!pip install -q --no-cache-dir \
  "numpy==2.2.6" \
  "torch==2.8.0" \
  "torchvision==0.23.0" \
  "torchaudio==2.8.0" \
  "torchcodec==0.7.*" \
  "pyannote.audio==3.4.0" \
  "pyannote.metrics==3.2.1" \
  scipy scikit-learn pandas

# 3) CAM++ via modelscope — PINNED to known-working versions (avoid drift)
#    These are the versions that worked in our dev_smoke run on 2026-05-25
!pip install -q addict simplejson sortedcontainers oss2 soundfile rotary_embedding_torch kaldiio
!pip install -q --no-cache-dir "datasets==2.21.0"   # modelscope 1.20.x needs datasets<3.0
!pip install -q "modelscope==1.20.1" "funasr==1.3.3"
!apt-get install -y libsndfile1 ffmpeg > /dev/null 2>&1

print()
print('=' * 78)
print('⚠️  REQUIRED: Runtime → Restart session（不要按 Restart and run all）')
print()
print('After restart, re-run THESE cells in order:')
print('  1. Drive mount + paths cell (cell 2)            ← variables lost after restart')
print('  2. HF_TOKEN cell (cell 3)                       ← variables lost after restart')
print('  3. Sanity check cell (cell 5, next after this) ← verify install + patches + CAM++')
print('  4. HF login cell (cell 6)')
print('  5. Helpers cell (cell 8)')
print('  6. Model runs (cell 11, 13, ...)')
print()
print('   Skip ONLY this install cell (cell 4).')
print('=' * 78)

In [ ]:
# === STEP 2: Sanity check — run this AFTER Runtime → Restart session ===
# 重啟後 Python kernel 會用新裝的 numpy/torch 二進位，不會再撞 binary mismatch
import sys

import numpy as np, torch, torchaudio, torchvision
print(f'numpy:       {np.__version__}')
print(f'torch:       {torch.__version__}')
print(f'torchaudio:  {torchaudio.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'AudioMetaData attribute present: {hasattr(torchaudio, "AudioMetaData")}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  Device: {torch.cuda.get_device_name()}')

# === Apply compat patches once, here, so all subsequent model loads inherit them ===
# Patch 1: pyannote 3.4 ↔ huggingface_hub use_auth_token incompatibility
import huggingface_hub.utils._validators as _v
if not getattr(_v, '_patched_for_pyannote', False):
    _orig_smoothly = _v.smoothly_deprecate_legacy_arguments
    def _patched_smoothly(fn_name, kwargs):
        kwargs = _orig_smoothly(fn_name, kwargs)
        if 'use_auth_token' in kwargs:
            tok = kwargs.pop('use_auth_token')
            if tok is not None and 'token' not in kwargs:
                kwargs['token'] = tok
        return kwargs
    _v.smoothly_deprecate_legacy_arguments = _patched_smoothly
    _v._patched_for_pyannote = True

# Patch 2: torch.load weights_only=True default vs legacy ckpts
if not getattr(torch, '_load_patched_for_legacy', False):
    _orig_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._load_patched_for_legacy = True

from pyannote.audio import Pipeline
print('\n✓ pyannote.audio import OK + compat patches applied — ready to run models')

# === Smoke check: CAM++ pipeline can actually instantiate (modelscope import path) ===
from modelscope.pipelines import pipeline as ms_pipeline
from modelscope.utils.constant import Tasks
print('✓ modelscope.pipelines import OK')

arr = np.array([1.0, 2.0, 3.0])
assert arr.sum() == 6.0 and (arr * 2).tolist() == [2.0, 4.0, 6.0], 'numpy ops broken'
print('✓ numpy ops OK')

In [ ]:
# === Login HF + setup model cache on Drive ===
from huggingface_hub import login
login(token=HF_TOKEN)

os.environ['HF_HOME'] = str(MODELS_CACHE / 'hf')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(MODELS_CACHE / 'hf')
os.environ['TRANSFORMERS_CACHE'] = str(MODELS_CACHE / 'hf')
print(f'HF cache → {os.environ["HF_HOME"]}')
print('✓ Logged in to HuggingFace')

## 1. Helpers — GT loader + DER computation

In [ ]:
import time, csv
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate
from tqdm.auto import tqdm

def load_rttm_to_annotation(rttm_path, uri):
    ann = Annotation(uri=uri)
    for line in rttm_path.read_text().strip().split('\n'):
        parts = line.split()
        if line.startswith('SPEAKER') and len(parts) >= 8:
            start, dur, spk = float(parts[3]), float(parts[4]), parts[7]
            ann[Segment(start, start + dur)] = spk
    return ann

def to_annotation(case_id, segs_list):
    ann = Annotation(uri=case_id)
    for s, e, spk in segs_list:
        if e > s:
            ann[Segment(float(s), float(e))] = str(spk)
    return ann

der_strict = DiarizationErrorRate(collar=0.0)
der_lenient = DiarizationErrorRate(collar=0.25)
jer = JaccardErrorRate(collar=0.25)

def expected_n_speakers(case_id):
    """From spec.participants — usually 2 (agent+customer), 3 if 'other' exists."""
    return DIALOG_META[case_id]['expected_n_speakers']

def eval_one(case_id, hyp_ann):
    """Compute metrics for one case against speech GT (VAD-trimmed)."""
    gt = load_rttm_to_annotation(GT_SPEECH / f'{case_id}.rttm', case_id)
    n_pred = len(set(hyp_ann.labels()))
    n_exp = expected_n_speakers(case_id)
    return {
        'case_id': case_id,
        'n_speakers_pred': n_pred,
        'n_speakers_expected': n_exp,
        'speaker_count_correct': n_pred == n_exp,
        'der_strict': round(der_strict(gt, hyp_ann) * 100, 2),
        'der_lenient_025': round(der_lenient(gt, hyp_ann) * 100, 2),
        'jer': round(jer(gt, hyp_ann) * 100, 2),
    }

def save_detection(model_id, case_id, hyp_ann, elapsed):
    d = DETECT_DIR / model_id
    d.mkdir(parents=True, exist_ok=True)
    segs = [{'start': float(s.start), 'end': float(s.end), 'speaker': str(l)}
            for s, _, l in hyp_ann.itertracks(yield_label=True)]
    (d / f'{case_id}.json').write_text(json.dumps({
        'case_id': case_id, 'model_id': model_id, 'elapsed_sec': elapsed,
        'segments': segs,
    }, ensure_ascii=False, indent=2))

def load_cached(model_id, case_id):
    """Used for ERes2NetV2 sanity check vs CAM++ only. Production calls clear cache at startup
       (see setup cell), so this returns None during normal model runs."""
    p = DETECT_DIR / model_id / f'{case_id}.json'
    if not p.exists(): return None
    cached = json.loads(p.read_text())
    ann = Annotation(uri=case_id)
    for s in cached['segments']:
        ann[Segment(s['start'], s['end'])] = s['speaker']
    return ann, cached.get('elapsed_sec', 0)

def run_on_all(model_id, infer_fn):
    """Runs inference on all 234 cases. FATAL on any failure or count mismatch.
       (Cache was archived at startup, so this is always fresh inference.)"""
    rows, failed = [], []
    for cid in tqdm(all_case_ids, desc=model_id):
        cached = load_cached(model_id, cid)
        if cached:
            hyp_ann, elapsed = cached
        else:
            try:
                t0 = time.time()
                hyp_ann = infer_fn(cid)
                elapsed = time.time() - t0
                save_detection(model_id, cid, hyp_ann, elapsed)
            except Exception as e:
                failed.append((cid, type(e).__name__, str(e)[:300]))
                continue
        row = eval_one(cid, hyp_ann)
        meta = DIALOG_META[cid]
        row.update({'model_id': model_id, 'subtype': meta['slice'],
                    'language': meta['language'], 'elapsed_sec': round(elapsed, 2)})
        rows.append(row)

    # === HARD INTEGRITY GATES ===
    if failed:
        print(f'\n❌ [{model_id}] {len(failed)} cases FAILED:')
        for cid, etype, emsg in failed[:10]:
            print(f'    {cid}: {etype}: {emsg}')
        if len(failed) > 10:
            print(f'    ... and {len(failed) - 10} more')
        raise RuntimeError(f'{model_id} produced {len(failed)}/{EXPECTED_N} failures; cannot trust subset benchmark')
    assert len(rows) == EXPECTED_N, f'{model_id} expected {EXPECTED_N} rows, got {len(rows)}'
    detection_files = list((DETECT_DIR / model_id).glob('*.json'))
    assert len(detection_files) == EXPECTED_N, f'{model_id} expected {EXPECTED_N} detection JSONs, got {len(detection_files)}'

    out = RESULTS_DIR / f'metrics_{model_id}.csv'
    with out.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    avg_der = sum(r['der_lenient_025'] for r in rows) / len(rows)
    spk_acc = sum(r['speaker_count_correct'] for r in rows) / len(rows) * 100
    print(f'  ✓ [{model_id}] {len(rows)}/{EXPECTED_N} | DER={avg_der:.2f}% | spk_acc={spk_acc:.0f}%')
    return rows
print('Helpers ready ✓')

## 2. Run 3 model configs

### 2.1 pyannote 3.1 (auto + fixed-N per spec)

In [ ]:
# === Patch 1: pyannote 3.4 內部用 use_auth_token，新版 huggingface_hub 拒收 ===
import huggingface_hub.utils._validators as _v
if not getattr(_v, '_patched_for_pyannote', False):
    _orig_smoothly = _v.smoothly_deprecate_legacy_arguments
    def _patched_smoothly(fn_name, kwargs):
        kwargs = _orig_smoothly(fn_name, kwargs)
        if 'use_auth_token' in kwargs:
            tok = kwargs.pop('use_auth_token')
            if tok is not None and 'token' not in kwargs:
                kwargs['token'] = tok
        return kwargs
    _v.smoothly_deprecate_legacy_arguments = _patched_smoothly
    _v._patched_for_pyannote = True
    print('✓ Patched huggingface_hub to swallow `use_auth_token`')

# === Patch 2: PyTorch 2.6+ weights_only=True 預設 vs pyannote/speechbrain 舊 ckpt ===
# 兩層保險：(a) allowlist 已知 safe globals  (b) force weights_only=False 在所有 entry point
import torch, torch.serialization

# Layer (a): allowlist the specific class the error mentioned + extras commonly needed
_safe = []
try: _safe.append(torch.torch_version.TorchVersion)
except Exception: pass
try:
    import numpy as _np
    _safe += [_np.core.multiarray.scalar, _np.core.multiarray._reconstruct, _np.ndarray, _np.dtype]
except Exception: pass
if _safe:
    torch.serialization.add_safe_globals(_safe)
    print(f'✓ Allowlisted {len(_safe)} safe globals for weights_only loading')

# Layer (b): force weights_only=False at BOTH torch.load AND torch.serialization.load
# (speechbrain imports torch.serialization.load directly, bypassing torch.load patch)
if not getattr(torch, '_load_patched_v2', False):
    _orig_ser_load = torch.serialization.load
    def _forced_load(*args, **kwargs):
        kwargs['weights_only'] = False   # FORCE (not setdefault — caller might pass True)
        return _orig_ser_load(*args, **kwargs)
    torch.serialization.load = _forced_load
    torch.load = _forced_load            # also rebind torch.load to the patched one
    torch._load_patched_v2 = True
    print('✓ Force-patched torch.load + torch.serialization.load → weights_only=False')

# === Now actually load pyannote 3.1 ===
from pyannote.audio import Pipeline

def get_annotation(result):
    return result.speaker_diarization if hasattr(result, 'speaker_diarization') else result

pyannote_31 = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1')
if torch.cuda.is_available():
    pyannote_31.to(torch.device('cuda'))
print('pyannote 3.1 loaded')

def infer_31_auto(cid):
    return get_annotation(pyannote_31(str(AUDIO_DIR / f'{cid}.wav')))
run_on_all('pyannote_31_auto', infer_31_auto)

def infer_31_fixedN(cid):
    n = expected_n_speakers(cid)   # 2 or 3 per spec.participants
    return get_annotation(pyannote_31(str(AUDIO_DIR / f'{cid}.wav'), num_speakers=n))
run_on_all('pyannote_31_fixedN', infer_31_fixedN)

del pyannote_31
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

### 2.2 3D-Speaker CAM++

In [ ]:
from modelscope.pipelines import pipeline as ms_pipeline
from modelscope.utils.constant import Tasks

campp = ms_pipeline(task=Tasks.speaker_diarization,
                     model='iic/speech_campplus_speaker-diarization_common')
print('CAM++ loaded')

def campp_to_annotation(case_id, output):
    if isinstance(output, dict) and 'text' in output:
        segs = [(float(e[0]), float(e[1]), f'spk_{int(e[2])}') for e in output['text']]
        return to_annotation(case_id, segs)
    return Annotation(uri=case_id)

def infer_campp(cid):
    return campp_to_annotation(cid, campp(str(AUDIO_DIR / f'{cid}.wav')))

run_on_all('campp_3dspeaker', infer_campp)

## 3. Cross-model summary

In [ ]:
import pandas as pd

MODELS = [
    ('pyannote_31_auto',     'pyannote 3.1 (auto)'),
    ('pyannote_31_fixedN',   'pyannote 3.1 (fixed N)'),
    ('campp_3dspeaker',      'CAM++'),
]

all_dfs = {}
for model_id, label in MODELS:
    p = RESULTS_DIR / f'metrics_{model_id}.csv'
    if p.exists():
        all_dfs[label] = pd.read_csv(p)
    else:
        print(f'  {label}: not run yet')

print(f'\n=== Overall Summary ({len(all_case_ids)} dialogs) ===\n')
print(f'{"Model":24} {"DER lenient":>14} {"DER strict":>12} {"JER":>8} {"spk acc":>8} {"avg time":>10}')
print('-' * 90)
summary = []
for label, df in all_dfs.items():
    der_l = df['der_lenient_025'].mean()
    der_l_std = df['der_lenient_025'].std()
    der_s = df['der_strict'].mean()
    jer_v = df['jer'].mean()
    spk_acc = df['speaker_count_correct'].mean() * 100
    avg_time = df['elapsed_sec'].mean()
    print(f'{label:24}  {der_l:>8.2f}±{der_l_std:.1f}%  {der_s:>10.2f}%  {jer_v:>6.2f}%  {spk_acc:>6.1f}%  {avg_time:>7.2f}s')
    summary.append({'model': label, 'der_lenient': der_l, 'der_lenient_std': der_l_std,
                    'der_strict': der_s, 'jer': jer_v, 'spk_acc': spk_acc, 'avg_time_sec': avg_time})

pd.DataFrame(summary).to_csv(RESULTS_DIR / 'cross_model_summary.csv', index=False)
print(f'\n→ {RESULTS_DIR / "cross_model_summary.csv"}')

In [ ]:
# === Per-slice DER ===
print('=== DER by slice (per model) ===\n')
rows = []
for label, df in all_dfs.items():
    by_slice = df.groupby('subtype')['der_lenient_025'].mean().to_dict()
    rows.append({'model': label, **by_slice})
by_slice_df = pd.DataFrame(rows).set_index('model').round(2)
print(by_slice_df)
by_slice_df.to_csv(RESULTS_DIR / 'der_by_slice.csv')

print('\n=== DER by language ===\n')
rows = []
for label, df in all_dfs.items():
    by_lang = df.groupby('language')['der_lenient_025'].mean().to_dict()
    rows.append({'model': label, **by_lang})
by_lang_df = pd.DataFrame(rows).set_index('model').round(2)
print(by_lang_df)
by_lang_df.to_csv(RESULTS_DIR / 'der_by_language.csv')

In [ ]:
# === Pyannote auto vs fixed-N (per spec) ===
print('=== Pyannote auto vs fixed N ===\n')
for family_prefix, family_name in [('pyannote 3.1', 'pyannote 3.1'), ('community-1', 'community-1')]:
    auto_label = f'{family_prefix} (auto)'
    fixed_label = f'{family_prefix} (fixed N)'
    if auto_label in all_dfs and fixed_label in all_dfs:
        a = all_dfs[auto_label]
        f = all_dfs[fixed_label]
        print(f'{family_name}:')
        print(f'  auto:  DER {a["der_lenient_025"].mean():.2f}%, spk_acc {a["speaker_count_correct"].mean()*100:.0f}%')
        print(f'  n=2:   DER {f["der_lenient_025"].mean():.2f}%, spk_acc {f["speaker_count_correct"].mean()*100:.0f}%')
        delta = a['der_lenient_025'].mean() - f['der_lenient_025'].mean()
        print(f'  → n=2 wins by {delta:.2f}pp\n')

In [ ]:
# === Statistical significance: best model vs CAM++ ===
from scipy import stats
if 'CAM++' in all_dfs:
    campp_df = all_dfs['CAM++']
    print('=== Paired t-test vs CAM++ ===\n')
    print(f'{"Comparison":40} {"t":>7} {"p":>8} {"verdict":<20}')
    print('-' * 80)
    for label, df in all_dfs.items():
        if label == 'CAM++': continue
        merged = campp_df.merge(df, on='case_id', suffixes=('_c', '_x'))
        if len(merged) < 2: continue
        t, p = stats.ttest_rel(merged['der_lenient_025_c'], merged['der_lenient_025_x'])
        delta = merged['der_lenient_025_x'].mean() - merged['der_lenient_025_c'].mean()
        verdict = 'CAM++ better' if delta > 0 and p < 0.05 else ('CAM++ worse' if delta < 0 and p < 0.05 else 'no sig diff')
        print(f'CAM++ vs {label:33} {t:>+6.2f}  {p:>7.4f}  {verdict}')

In [ ]:
# === Final recommendation ===
if all_dfs:
    best_label, best_df = min(all_dfs.items(), key=lambda kv: kv[1]['der_lenient_025'].mean())
    der = best_df['der_lenient_025'].mean()
    print(f'=== Best model: {best_label} ===')
    print(f'  Mean DER (lenient): {der:.2f}%')
    print(f'  Mean DER (strict):  {best_df["der_strict"].mean():.2f}%')
    print(f'  Speaker count acc:  {best_df["speaker_count_correct"].mean()*100:.0f}%\n')
    
    if der < 8:    rec = '🟢 PRODUCTION READY — top-tier'
    elif der < 12: rec = '🟢 PRODUCTION READY — within accepted band'
    elif der < 18: rec = '🟡 ACCEPTABLE for offline analytics; tune before live'
    elif der < 25: rec = '🟠 PoC ONLY — needs real-data fine-tune'
    else:          rec = '🔴 NOT VIABLE — investigate'
    print(rec)
    
    print(f'\n→ Results saved to: {RESULTS_DIR}')
    print(f'  • metrics_{{model_id}}.csv  ({len(all_dfs)} files)')
    print(f'  • cross_model_summary.csv')
    print(f'  • der_by_slice.csv, der_by_language.csv')
    print(f'  • detections/{{model_id}}/{{case_id}}.json')
    print(f'\n下載這些檔案做 v2 final report.')